# MetAgeFormer — 3. Metabolic Subtype Assignment

Assign **metabolic subtypes** (Leiden clusters 0–12) and **meta-subtypes** (1–4) to
samples from their 512-dim metabolomic embeddings, using the released focal-loss MLP
classifier.

**Requirements**
- `Model_Weights/SubtypeClassifier/subtype_mlp_classifier_focal.joblib` (focal MLP classifier)
- `Model_Weights/MetAgeFormer/` (backbone: `config.json`, `tokenizer.pkl`, `model_weights.pth`)
  — embeddings must come from this released backbone (the classifier was trained on its
  embedding space; the classifier input is the **raw** 512-dim embedding, no scaling)
- An AnnData NMR dataset with layer `Z-score normalized` (or any source of
  `(n_samples, 512)` embeddings)


In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "Src"))

import numpy as np
import pandas as pd
import torch

from metageformer_torch.subtype_mlp import FocalMLPClassifier

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


## 1. Load the subtype classifier

The model file carries a `.joblib` extension but is a PyTorch `torch.save` archive —
load it with `FocalMLPClassifier.load()`, never `joblib.load()`.


In [ ]:
CLASSIFIER_PATH = REPO_ROOT / "Model_Weights" / "SubtypeClassifier" / "subtype_mlp_classifier_focal.joblib"
assert CLASSIFIER_PATH.exists(), "Missing subtype classifier — see Model_Weights/readme.md"

classifier = FocalMLPClassifier.load(str(CLASSIFIER_PATH), device=DEVICE)
print("classes (Leiden clusters):", classifier.classes_.tolist())
print("class counts (training):", classifier.class_counts_)


## 2. Meta-subtype mapping

The 13 Leiden clusters are grouped into 4 **meta-subtypes** by the manual mapping used
in the paper (first match wins):


In [ ]:
META_SUBTYPES = {
    "Meta-subtype 1": [8],
    "Meta-subtype 2": [0, 5, 2, 3],
    "Meta-subtype 3": [6, 9, 12, 7, 10],
    "Meta-subtype 4": [4, 1, 11],
}

def subtype_to_meta(subtype_array):
    """Map cluster labels (0-12) to meta-subtype numbers (1-4); NaN if unmapped."""
    lookup = {}
    for meta_num, (name, clusters) in enumerate(META_SUBTYPES.items(), start=1):
        for c in clusters:
            lookup[int(c)] = meta_num
    out = np.array([lookup.get(int(s), np.nan) for s in subtype_array])
    return out

pd.DataFrame(
    [{"meta-subtype": name, "clusters": clusters} for name, clusters in META_SUBTYPES.items()]
)


## 3. Extract embeddings with the released backbone

Same flow as Notebook 1 — tokenize (NaN-aware) and run the backbone. You can also pass
any `(n_samples, 512)` float32 matrix of embeddings produced by this backbone.


In [ ]:
import anndata as ad
from utils import load_tokenizer
from metageformer_torch.models import MetAgeFormer_Pretrained

tokenizer = load_tokenizer(str(REPO_ROOT / "Model_Weights/MetAgeFormer/tokenizer.pkl"))
with open(REPO_ROOT / "Model_Weights/MetAgeFormer/config.json") as f:
    model_config = json.load(f)

embedding_module_conf = {"n_vocabs": {"identifier": tokenizer.vocab_size_identifiers}}
model = MetAgeFormer_Pretrained(embedding_module_conf, model_config,
                                str(REPO_ROOT / "Model_Weights/MetAgeFormer/model_weights.pth"))
model.to(DEVICE).eval()

DATA_PATH = REPO_ROOT / "Data" / "NMR_dataset_fullcohort_107nonderived" / "val.h5ad"
# Fake-data demo: DATA_PATH = REPO_ROOT / "Data" / "fake" / "NMR_dataset_fake" / "val.h5ad"

adata = ad.read_h5ad(DATA_PATH)[:64]
inputs, _ = tokenizer.tokenize_from_anndata(
    adata, padding="longest", masking="missing",
    data_layer="Z-score normalized", mode="inference", return_tensor=True, device=DEVICE)
with torch.inference_mode():
    embs = model(inputs)["embs"].cpu().numpy().astype(np.float32)
print("embeddings:", embs.shape)


## 4. Assign subtypes and meta-subtypes


In [ ]:
subtype = classifier.predict(embs)              # cluster labels 0-12
proba = classifier.predict_proba(embs)           # (n_samples, 13), columns = clusters 0..12
meta = subtype_to_meta(subtype)                  # meta-subtype numbers 1-4

results = pd.DataFrame({
    "subtype": subtype,
    "meta_subtype": meta,
    "confidence": proba.max(axis=1),
})
print("Subtype counts:")
print(results["subtype"].value_counts().sort_index().to_string())
print("\nMeta-subtype counts:")
print(results["meta_subtype"].value_counts().sort_index().to_string())
results.head()


## Notes

- Classifier input: **raw** 512-dim embeddings (no normalization/scaling), exactly as
  produced by the released backbone. `predict_proba` is a softmax over the 13 clusters.
- The `.joblib` file is a torch archive (see cell 1); inference needs only `torch` + `numpy`.
- For training the classifier (or the sklearn variant) see the paper's repository.
- Optional visualization: `matplotlib` (not required).
